# Stage 03 — Replication

Reproduce the paper's main OLS/IV regressions. Write JSON + LaTeX table.
Follow `skills/stage_03.md` for detailed instructions.

In [1]:
from paths import *
import json
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

df   = pd.read_parquet(str(DATASET_PATH))
spec = json.loads(PAPER_SPEC.read_text())

outcome   = spec['identification']['outcome_variable']
treatment = spec['identification']['treatment_variable']
instrument = spec['identification'].get('instrument')
controls  = spec['identification']['controls']
id_type   = spec['identification']['type']

print(f'Identification: {id_type}')
print(f'N obs (complete): {df[[outcome, treatment]].dropna().shape[0]}')

Identification: OLS
N obs (complete): 63


In [2]:
# --- Run 3 OLS replications (Table 2 Panels A, B, C) ---
# Each panel: growth ~ treatment + 17 controls, HC1 robust SEs, N=63

treatments_info = [
    {
        "treatment": "skill1_corr",
        "panel": "Table 2, Panel A",
        "spec": "OLS — growth on skill1_corr, 17 controls, country-level",
        "published_coef": 0.035,
        "published_se": 0.010,
    },
    {
        "treatment": "diffa",
        "panel": "Table 2, Panel B",
        "spec": "OLS — growth on diffa, 17 controls, country-level",
        "published_coef": 0.016,
        "published_se": 0.006,
    },
    {
        "treatment": "diffg",
        "panel": "Table 2, Panel C",
        "spec": "OLS — growth on diffg, 17 controls, country-level",
        "published_coef": 0.020,
        "published_se": 0.004,
    },
]

replication_specs = []
models = {}  # store fitted models for LaTeX table

for info in treatments_info:
    t = info["treatment"]
    X = sm.add_constant(df[controls + [t]])
    y = df.loc[X.index, outcome]
    model = sm.OLS(y, X).fit(cov_type="HC1")
    models[t] = model

    rep_coef = float(model.params[t])
    rep_se = float(model.bse[t])
    pub_coef = info["published_coef"]
    pub_se = info["published_se"]
    abs_diff = abs(rep_coef - pub_coef)
    rel_diff = abs_diff / abs(pub_coef) * 100 if pub_coef != 0 else float("nan")

    replication_specs.append({
        "spec": info["spec"],
        "table": info["panel"],
        "published_coef": pub_coef,
        "replicated_coef": rep_coef,
        "abs_diff": abs_diff,
        "rel_diff_pct": round(rel_diff, 2),
        "published_se": pub_se,
        "replicated_se": rep_se,
        "match": rel_diff < 5.0,
        "threshold_pct": 5.0,
        "n_obs": int(model.nobs),
        "r_squared": float(model.rsquared),
        "pvalue": float(model.pvalues[t]),
    })

    print(f"{info['panel']:20s}  treatment={t}")
    print(f"  Published : coef={pub_coef:.4f}  SE={pub_se:.4f}")
    print(f"  Replicated: coef={rep_coef:.6f}  SE={rep_se:.6f}")
    print(f"  Rel diff  : {rel_diff:.2f}%  {'PASS' if rel_diff < 5.0 else 'FAIL'}")
    print()

Table 2, Panel A      treatment=skill1_corr
  Published : coef=0.0350  SE=0.0100
  Replicated: coef=0.034839  SE=0.012184
  Rel diff  : 0.46%  PASS

Table 2, Panel B      treatment=diffa
  Published : coef=0.0160  SE=0.0060
  Replicated: coef=0.016262  SE=0.005716
  Rel diff  : 1.64%  PASS

Table 2, Panel C      treatment=diffg
  Published : coef=0.0200  SE=0.0040
  Replicated: coef=0.018978  SE=0.004935
  Rel diff  : 5.11%  FAIL



In [3]:
# --- Summary comparison table ---
print(f"{'Panel':<22s} {'Published':>10s} {'Replicated':>12s} {'Rel Diff %':>10s} {'Status':>8s}")
print("-" * 66)
for r in replication_specs:
    status = "PASS" if r["match"] else "FAIL"
    print(f"{r['table']:<22s} {r['published_coef']:>10.4f} {r['replicated_coef']:>12.6f} {r['rel_diff_pct']:>10.2f} {status:>8s}")
print()
n_pass = sum(r["match"] for r in replication_specs)
print(f"Overall: {n_pass}/{len(replication_specs)} specifications replicated within 5% threshold")

Panel                   Published   Replicated Rel Diff %   Status
------------------------------------------------------------------
Table 2, Panel A           0.0350     0.034839       0.46     PASS
Table 2, Panel B           0.0160     0.016262       1.64     PASS
Table 2, Panel C           0.0200     0.018978       5.11     FAIL

Overall: 2/3 specifications replicated within 5% threshold


In [4]:
# --- Write replication_results.json ---
n_pass = sum(r['match'] for r in replication_specs)
replication_results = {
    'specs': replication_specs,
    'overall_pass': n_pass == len(replication_specs),
    'n_specs': len(replication_specs),
    'n_pass': n_pass
}
REPLICATION_RESULTS.write_text(json.dumps(replication_results, indent=2))

worst = max((r['rel_diff_pct'] for r in replication_specs), default=0)
replication_check = {
    'pass': replication_results['overall_pass'],
    'summary': f"{n_pass}/{len(replication_specs)} specs replicated within 5%",
    'worst_rel_diff_pct': worst
}

# If any fail, add failure notes
if not replication_results['overall_pass']:
    failed = [r for r in replication_specs if not r['match']]
    replication_check['failure_notes'] = (
        f"{len(failed)} spec(s) exceeded 5% threshold: "
        + "; ".join(f"{r['table']} ({r['rel_diff_pct']:.2f}%)" for r in failed)
    )

REPLICATION_CHECK.write_text(json.dumps(replication_check, indent=2))
print(f"Wrote: {REPLICATION_RESULTS}")
print(f"Wrote: {REPLICATION_CHECK}")
print()
print(json.dumps(replication_check, indent=2))

Wrote: C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\data\results\replication_results.json
Wrote: C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\data\results\replication_check.json

{
  "pass": false,
  "summary": "2/3 specs replicated within 5%",
  "worst_rel_diff_pct": 5.11,
  "failure_notes": "1 spec(s) exceeded 5% threshold: Table 2, Panel C (5.11%)"
}


In [5]:
# --- Write LaTeX replication table ---

def fmt_coef(val):
    """Format coefficient to 3 decimal places."""
    return f"{val:.3f}"

def fmt_se(val):
    """Format SE in parentheses."""
    return f"({val:.3f})"

def stars(pval):
    """Significance stars."""
    if pval < 0.01:
        return "***"
    elif pval < 0.05:
        return "**"
    elif pval < 0.10:
        return "*"
    return ""

panels = [
    ("Panel A: Skill-tariff correlation (skill1\\_corr)", "skill1_corr", 0),
    ("Panel B: Tariff differential, low cut-off (diffa)", "diffa", 1),
    ("Panel C: Tariff differential, high cut-off (diffg)", "diffg", 2),
]

lines = []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\caption{Replication of Nunn \& Trefler (2010) Table 2, Column 8}")
lines.append(r"\label{tab:replication}")
lines.append(r"\begin{tabular}{lcc}")
lines.append(r"\hline\hline")
lines.append(r" & Published & Replicated \\")
lines.append(r"\hline")

for panel_label, tvar, idx in panels:
    r = replication_specs[idx]
    m = models[tvar]
    p_stars = stars(r["pvalue"])

    lines.append(f"\\multicolumn{{3}}{{l}}{{\\textit{{{panel_label}}}}} \\\\[3pt]")
    lines.append(f"Treatment & {fmt_coef(r['published_coef'])} & {fmt_coef(r['replicated_coef'])}{p_stars} \\\\")
    lines.append(f" & ({fmt_coef(r['published_se'])}) & {fmt_se(r['replicated_se'])} \\\\")
    lines.append(f"$R^2$ & & {m.rsquared:.3f} \\\\")
    lines.append(f"$N$ & {r['n_obs']} & {r['n_obs']} \\\\")
    lines.append(r"\hline")

lines.append(r"Controls & Yes & Yes \\")
lines.append(r"SE type & Robust & HC1 \\")
lines.append(r"\hline\hline")
lines.append(r"\multicolumn{3}{p{0.7\textwidth}}{\footnotesize Notes: Published estimates from Nunn \& Trefler (2010) Table 2, Column 8. Replicated estimates use OLS with HC1 robust standard errors. All regressions include 17 controls (average tariff, initial GDP, investment, human capital, region dummies, period dummies, initial skill intensities). *$p<0.10$, **$p<0.05$, ***$p<0.01$.} \\")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

table_tex = "\n".join(lines)
table_path = TABLES_DIR / "table_replication.tex"
table_path.write_text(table_tex)
print(f"Wrote: {table_path}")
print()
print(table_tex)

Wrote: C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\paper\tables\table_replication.tex

\begin{table}[htbp]
\centering
\caption{Replication of Nunn \& Trefler (2010) Table 2, Column 8}
\label{tab:replication}
\begin{tabular}{lcc}
\hline\hline
 & Published & Replicated \\
\hline
\multicolumn{3}{l}{\textit{Panel A: Skill-tariff correlation (skill1\_corr)}} \\[3pt]
Treatment & 0.035 & 0.035*** \\
 & (0.010) & (0.012) \\
$R^2$ & & 0.708 \\
$N$ & 63 & 63 \\
\hline
\multicolumn{3}{l}{\textit{Panel B: Tariff differential, low cut-off (diffa)}} \\[3pt]
Treatment & 0.016 & 0.016*** \\
 & (0.006) & (0.006) \\
$R^2$ & & 0.700 \\
$N$ & 63 & 63 \\
\hline
\multicolumn{3}{l}{\textit{Panel C: Tariff differential, high cut-off (diffg)}} \\[3pt]
Treatment & 0.020 & 0.019*** \\
 & (0.004) & (0.005) \\
$R^2$ & & 0.748 \\
$N$ & 63 & 63 \\
\hline
Controls & Yes & Yes \\
SE type & Robust & HC1 \\
\hline\hline
\multicolumn{3}{p{0.7\textwidth}}{\footnotesize Notes: Published 